# 🧪 Lab 8: Navigating the Temporal Anomaly (Structured Streaming Tab)

In this laboratory session, we observe Spark Structured Streaming in **micro-batch mode** through the Spark Web UI.

**Objective:** Run controlled streaming workloads, inspect input and processing rates, compare batch duration with trigger intervals, add a stateful window with a watermark, observe state-store metrics, and connect slow micro-batches back to SQL, Stages, Executors, and logs.

> Note: this lab uses the `rate` source and local mode for repeatable demonstrations. Exact metrics depend on your machine, Spark version, local task slots, and runtime load.


## Step 1: Initialize the Temporal Sensor Array

We start a local Spark session, enable the Spark Web UI, request port `4040`, and print the **actual** UI URL.

Do not assume the UI is always on `4040`. If that port is busy, Spark may retry the next ports according to `spark.port.maxRetries`.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time
import tempfile
import shutil

# Stop any old active streaming queries/session from previous notebook runs.
existing = SparkSession.getActiveSession()
if existing is not None:
    for q in existing.streams.active:
        try:
            q.stop()
        except Exception as exc:
            print(f"Could not stop old query {q.name}: {exc}")
    existing.stop()

spark = (
    SparkSession.builder
    .appName("spark-ui-streaming-temporal-anomaly-lab-08")
    .master("local[*]")
    .config("spark.ui.enabled", "true")
    .config("spark.ui.port", "4040")
    .config("spark.port.maxRetries", "16")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

base_checkpoint_dir = tempfile.mkdtemp(prefix="spark_ui_lab08_checkpoints_")

print(f"Active Spark version: {spark.version}")
print(f"🛰️ Actual Spark UI: {spark.sparkContext.uiWebUrl}")
print(f"📦 Temporary checkpoint base: {base_checkpoint_dir}")


def print_stream_progress(query, label=None):
    """Print the most useful pieces of StreamingQueryProgress without assuming every metric exists."""
    p = query.lastProgress
    query_label = label or query.name or str(query.id)

    if not p:
        print(f"\n[{query_label}] No progress event yet.")
        return

    print(f"\n📊 [{query_label}] Micro-batch {p.get('batchId')}")
    print(f"  - Query ID: {query.id}")
    print(f"  - Run ID:   {query.runId}")
    print(f"  - Input rows: {p.get('numInputRows')}")
    print(f"  - Input rows/sec: {p.get('inputRowsPerSecond')}")
    print(f"  - Processed rows/sec: {p.get('processedRowsPerSecond')}")

    event_time = p.get("eventTime") or {}
    if event_time:
        print(f"  - Event-time watermark: {event_time.get('watermark')}")

    duration_ms = p.get("durationMs") or {}
    if duration_ms:
        print("  - Duration phases:")
        for phase, ms in sorted(duration_ms.items()):
            print(f"      {phase}: {ms} ms")

    state_ops = p.get("stateOperators") or []
    if state_ops:
        print("  - State operator metrics:")
        for idx, state in enumerate(state_ops):
            print(f"      Operator {idx}:")
            print(f"        rows total: {state.get('numRowsTotal')}")
            print(f"        rows updated: {state.get('numRowsUpdated')}")
            print(f"        rows removed: {state.get('numRowsRemoved')}")
            print(f"        memory used bytes: {state.get('memoryUsedBytes')}")
            print(f"        rows dropped by watermark: {state.get('numRowsDroppedByWatermark')}")


Active Spark version: 4.1.2
🛰️ Actual Spark UI: http://T14-PF4WM3XL.sdggroup.local:4040
📦 Temporary checkpoint base: C:\Users\ANGEL~1.ALV\AppData\Local\Temp\spark_ui_lab08_checkpoints_ds2uahu4


## Step 2: Start a Stateless Rate Stream

We create a micro-batch `rate` source and write it to the memory sink.

The memory sink is useful for demos and debugging, not production. The important thing here is that the stream appears in the **Structured Streaming** tab with a query name, query ID, run ID, input rate, processing rate, and batch progress.

`query.id` can persist across restarts when a query is restarted from the same checkpoint. `query.runId` changes for each start/restart.


In [2]:
stateless_source = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 200)
    .option("numPartitions", 4)
    .load()
)

stateless_lineage = (
    stateless_source
    .select(
        "timestamp",
        "value",
        (F.col("value") % 10).alias("sector_id")
    )
)

stateless_query = (
    stateless_lineage.writeStream
    .format("memory")
    .queryName("stateless_rate_sensor_stream")
    .outputMode("append")
    .option("checkpointLocation", f"{base_checkpoint_dir}/stateless")
    .trigger(processingTime="5 seconds")
    .start()
)

print("🛰️ Stateless rate stream launched.")
print(f"Name:   {stateless_query.name}")
print(f"ID:     {stateless_query.id}")
print(f"Run ID: {stateless_query.runId}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")


🛰️ Stateless rate stream launched.
Name:   stateless_rate_sensor_stream
ID:     adb5053d-b983-408b-9cd4-34bf5eeaa732
Run ID: 9de259a7-4675-4633-bf7a-17c6f4cad16d
Spark UI: http://T14-PF4WM3XL.sdggroup.local:4040


## Step 3: Read the Streaming Pulse

We wait for a few micro-batches and print the same kinds of signals you should inspect in the Structured Streaming tab:

- input rows
- input rows per second
- processed rows per second
- batch duration phases

A query can be active while falling behind. Active is a pulse, not a health certificate.


In [3]:
print("⏳ Gathering stateless stream telemetry...")

for check in range(4):
    time.sleep(5)
    print_stream_progress(stateless_query, "stateless_rate_sensor_stream")

print("\nSample rows currently visible through the memory sink:")
spark.sql("""
    SELECT *
    FROM stateless_rate_sensor_stream
    ORDER BY timestamp DESC
    LIMIT 5
""").show(truncate=False)


⏳ Gathering stateless stream telemetry...

📊 [stateless_rate_sensor_stream] Micro-batch 0
  - Query ID: adb5053d-b983-408b-9cd4-34bf5eeaa732
  - Run ID:   9de259a7-4675-4633-bf7a-17c6f4cad16d
  - Input rows: 0
  - Input rows/sec: 0.0
  - Processed rows/sec: 0.0
  - Duration phases:
      addBatch: 2395 ms
      commitOffsets: 757 ms
      getBatch: 7 ms
      latestOffset: 0 ms
      queryPlanning: 789 ms
      triggerExecution: 4495 ms
      walCommit: 479 ms

📊 [stateless_rate_sensor_stream] Micro-batch 2
  - Query ID: adb5053d-b983-408b-9cd4-34bf5eeaa732
  - Run ID:   9de259a7-4675-4633-bf7a-17c6f4cad16d
  - Input rows: 400
  - Input rows/sec: 230.7
  - Processed rows/sec: 398.8
  - Duration phases:
      addBatch: 112 ms
      commitOffsets: 501 ms
      getBatch: 0 ms
      latestOffset: 0 ms
      queryPlanning: 11 ms
      triggerExecution: 1003 ms
      walCommit: 378 ms

📊 [stateless_rate_sensor_stream] Micro-batch 3
  - Query ID: adb5053d-b983-408b-9cd4-34bf5eeaa732
  - Run I

## Step 4: Add Stateful Cargo with a Watermark

Now we create a stateful streaming query: a windowed aggregation with a watermark.

Watermarks tell Spark when old event-time state may be eligible to be finalized, dropped, or cleaned up, depending on the stateful operator and output mode. They are not just correctness decoration; they are also part of the state-management boundary.

This query uses `foreachBatch` as a lightweight demo sink. The action inside `foreachBatch` forces each micro-batch to execute so the UI has real work and state-store metrics to show.


In [4]:
stateful_source = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 500)
    .option("numPartitions", 4)
    .load()
)

stateful_lineage = (
    stateful_source
    .withWatermark("timestamp", "10 seconds")
    .groupBy(
        F.window("timestamp", "10 seconds").alias("event_window"),
        (F.col("value") % 10).alias("galaxy_code")
    )
    .count()
)

stateful_batch_audit = []

def audit_stateful_batch(batch_df, batch_id):
    # This action materializes the output of the micro-batch.
    output_rows = batch_df.count()
    stateful_batch_audit.append((batch_id, output_rows))
    print(f"🧮 foreachBatch sink observed batch {batch_id} with {output_rows} output rows.")

stateful_query = (
    stateful_lineage.writeStream
    .outputMode("update")
    .option("checkpointLocation", f"{base_checkpoint_dir}/stateful")
    .trigger(processingTime="5 seconds")
    .foreachBatch(audit_stateful_batch)
    .start()
)

print("🌀 Stateful watermark/window stream launched.")
print(f"Name:   {stateful_query.name}")
print(f"ID:     {stateful_query.id}")
print(f"Run ID: {stateful_query.runId}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")


🌀 Stateful watermark/window stream launched.
Name:   None
ID:     de78f0d4-9a63-403a-a2f8-d91099de2f35
Run ID: 9e2c966e-6ab2-4a18-82b3-49319fe339d6
Spark UI: http://T14-PF4WM3XL.sdggroup.local:4040


## Step 5: Inspect Watermark and State Metrics

Watch for:

- event-time watermark
- state rows total
- state rows updated
- state memory used
- rows removed or dropped by watermark, when applicable

With the `rate` source, data normally arrives on time, so rows dropped by watermark may remain zero. That is fine. The goal is to learn where the state metrics live and how to read them.


In [5]:
print("⏳ Gathering stateful stream telemetry...")

for check in range(6):
    time.sleep(5)
    print_stream_progress(stateful_query, "stateful_watermark_window_stream")

print("\nStateful sink audit captured by foreachBatch:")
for batch_id, output_rows in stateful_batch_audit[-10:]:
    print(f"  - batch {batch_id}: {output_rows} output rows")


⏳ Gathering stateful stream telemetry...

[stateful_watermark_window_stream] No progress event yet.
🧮 foreachBatch sink observed batch 0 with 0 output rows.
🧮 foreachBatch sink observed batch 1 with 20 output rows.

📊 [stateful_watermark_window_stream] Micro-batch 1
  - Query ID: de78f0d4-9a63-403a-a2f8-d91099de2f35
  - Run ID:   9e2c966e-6ab2-4a18-82b3-49319fe339d6
  - Input rows: 3000
  - Input rows/sec: 511.0
  - Processed rows/sec: 887.6
  - Event-time watermark: 1970-01-01T00:00:00.000Z
  - Duration phases:
      addBatch: 2037 ms
      commitOffsets: 560 ms
      getBatch: 0 ms
      latestOffset: 0 ms
      queryPlanning: 51 ms
      triggerExecution: 3379 ms
      walCommit: 724 ms
  - State operator metrics:
      Operator 0:
        rows total: 20
        rows updated: 20
        rows removed: 0
        memory used bytes: 5984
        rows dropped by watermark: 0
🧮 foreachBatch sink observed batch 2 with 10 output rows.

📊 [stateful_watermark_window_stream] Micro-batch 2
  - 

## Step 6: Optional Slow Sink Anomaly

This small query intentionally sleeps inside `foreachBatch`.

It demonstrates a simple truth: a short trigger interval does not make a slow batch faster. If the sink or batch work takes longer than the trigger interval, Spark cannot escape physics by scheduling more impatiently.

This is a demo of streaming delay mechanics, not a production design pattern.


In [6]:
slow_sink_audit = []

def slow_sink(batch_df, batch_id):
    rows = batch_df.count()
    time.sleep(4)  # Artificial sink delay for UI demonstration.
    slow_sink_audit.append((batch_id, rows))
    print(f"🐌 slow_sink batch {batch_id}: {rows} rows, plus artificial delay.")

slow_source = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 100)
    .option("numPartitions", 2)
    .load()
)

slow_query = (
    slow_source.writeStream
    .outputMode("append")
    .option("checkpointLocation", f"{base_checkpoint_dir}/slow_sink")
    .trigger(processingTime="2 seconds")
    .foreachBatch(slow_sink)
    .start()
)

print("🐌 Slow sink anomaly launched.")
print(f"ID:     {slow_query.id}")
print(f"Run ID: {slow_query.runId}")

for check in range(3):
    time.sleep(5)
    print_stream_progress(slow_query, "slow_sink_anomaly_stream")

slow_query.stop()
print("🛑 Slow sink anomaly stopped. It should remain visible as a completed streaming query in the UI.")


🐌 Slow sink anomaly launched.
ID:     f40a183f-a5cc-40ab-901f-4fb7fb8dc72c
Run ID: e9729570-802e-44be-a5d9-6eff48412faa
🧮 foreachBatch sink observed batch 7 with 10 output rows.

[slow_sink_anomaly_stream] No progress event yet.
🐌 slow_sink batch 0: 0 rows, plus artificial delay.
🧮 foreachBatch sink observed batch 8 with 20 output rows.

📊 [slow_sink_anomaly_stream] Micro-batch 0
  - Query ID: f40a183f-a5cc-40ab-901f-4fb7fb8dc72c
  - Run ID:   e9729570-802e-44be-a5d9-6eff48412faa
  - Input rows: 0
  - Input rows/sec: 0.0
  - Processed rows/sec: 0.0
  - Duration phases:
      addBatch: 4276 ms
      commitOffsets: 668 ms
      getBatch: 0 ms
      latestOffset: 0 ms
      queryPlanning: 6 ms
      triggerExecution: 5524 ms
      walCommit: 572 ms
🐌 slow_sink batch 1: 600 rows, plus artificial delay.
🧮 foreachBatch sink observed batch 9 with 10 output rows.

📊 [slow_sink_anomaly_stream] Micro-batch 1
  - Query ID: f40a183f-a5cc-40ab-901f-4fb7fb8dc72c
  - Run ID:   e9729570-802e-44be-a5d9

## Step 7: Cross-Tab Traversal

Use the Structured Streaming tab as the temporal radar, then pivot:

- **Streaming → SQL tab:** inspect the micro-batch execution plan.
- **Streaming → Stages and Tasks:** inspect task duration, shuffle, spill, and skew.
- **Streaming → Executors:** inspect GC, memory, failed tasks, logs, and executor loss.

The Streaming tab tells you the stream is falling behind. The other tabs usually tell you what hit the ship.


In [7]:
print(f"⏳ Explore the Structured Streaming tab now: {spark.sparkContext.uiWebUrl}")
print("Suggested checks:")
print("  1. Open the Structured Streaming tab.")
print("  2. Compare stateless, stateful, and slow-sink query rows.")
print("  3. Click each Run ID and inspect rates, batch duration, operation duration, watermark, and state metrics.")
print("  4. Pivot to SQL, Stages, Tasks, and Executors for the corresponding micro-batches.")

input("Press Enter when you are done inspecting the Spark UI and want to stop the streams...")


⏳ Explore the Structured Streaming tab now: http://T14-PF4WM3XL.sdggroup.local:4040
Suggested checks:
  1. Open the Structured Streaming tab.
  2. Compare stateless, stateful, and slow-sink query rows.
  3. Click each Run ID and inspect rates, batch duration, operation duration, watermark, and state metrics.
  4. Pivot to SQL, Stages, Tasks, and Executors for the corresponding micro-batches.
🧮 foreachBatch sink observed batch 10 with 20 output rows.


''

## Step 8: Stop Streams and Clean the Checkpoint Cargo Bay

We stop active streaming queries, remove temporary checkpoints, and stop Spark.

Once the SparkContext stops, the Live UI disappears. Post-mortem analysis requires persisted event logs and a Spark History Server.


In [8]:
for q in list(spark.streams.active):
    print(f"Stopping query: {q.name or q.id}")
    q.stop()

shutil.rmtree(base_checkpoint_dir, ignore_errors=True)
print("🧹 Temporary checkpoint directory cleaned up.")

spark.stop()
print("💀 Streaming queries stopped. SparkContext stopped. The Live UI is no longer available.")


Stopping query: stateless_rate_sensor_stream
Stopping query: de78f0d4-9a63-403a-a2f8-d91099de2f35
🧹 Temporary checkpoint directory cleaned up.
💀 Streaming queries stopped. SparkContext stopped. The Live UI is no longer available.


## 📊 Post-Lab Analysis: Evidence from the Temporal Anomaly

### 1. Active Is Not Healthy

The Structured Streaming overview shows whether a query is running, completed, or failed. That is only the pulse check.

A stream can remain active while input rate exceeds processing rate across sustained triggers. In that case, backlog usually grows, depending on source limits, trigger behavior, and unread available data.

### 2. Batch Duration Must Be Compared with Trigger Interval

If a query is triggered every 5 seconds and the micro-batch regularly takes less than 5 seconds, the stream has breathing room.

If a query is triggered every 2 seconds but each batch takes longer than that, the stream cannot magically obey the trigger interval. Lower latency settings do not make slow work faster.

### 3. Operation Duration Shows Where Time Went

Use operation duration phases such as `latestOffset`, `getBatch`, `queryPlanning`, `addBatch`, and `walCommit` to locate the pain.

High `latestOffset` or `getBatch` may point toward source or file-listing friction. High `walCommit` may point toward checkpoint storage latency. High `addBatch` usually means the actual processing and sink work dominated.

### 4. Watermarks Are State Boundaries, Not Decorations

Watermarks tell Spark when old event-time state may be eligible to be finalized, dropped, or cleaned up, depending on the stateful operator and output mode.

If state rows and state memory keep climbing, inspect the watermark, key distribution, state-store metrics, and the physical plan behind the micro-batches.

### 5. Query ID and Run ID Are Different

A query ID can persist across restarts when a query is restarted from the same checkpoint. A run ID changes for each start or restart.

Do not compare two different runs as if they were the same incarnation of the stream.

### ⏱️ Validation Summary

* **Step 1:** Started a local Spark session and printed the actual Spark Web UI URL.
* **Step 2:** Started a stateless micro-batch rate stream and inspected input/processing rates.
* **Step 3:** Started a stateful windowed aggregation with a watermark and inspected state-store metrics.
* **Step 4:** Created an optional slow-sink anomaly to compare batch duration with trigger interval.
* **Step 5:** Kept the Live UI open for cross-tab inspection, then stopped streams and cleaned temporary checkpoints.
